# Робастность запросов, ошибки и известность статей

Цель ноутбука — отделить реальные ограничения модели от общих впечатлений. Он отвечает на четыре вопроса:

1. На какой стадии теряются правильные статьи: retrieval, reranker или финальная голова?
2. Связаны ли ошибки с опечатками, длиной, плейсхолдерами, несколькими GT и редкими статьями?
3. Насколько надёжен дешёвый proxy-протокол с одной из пяти OOF-моделей?
4. Какие контрфактические преобразования запросов стоит отправить на GPU после завершения текущего обучения?

> **Ограничение:** ноутбук по умолчанию только читает готовые артефакты и работает на CPU. Он не импортирует `torch`, не создаёт CUDA-контекст и не запускает модели. Экспорт вариантов выключен флагом `EXPORT_VARIANTS=False`. GPU-протокол дан в конце только как спецификация будущего внешнего запуска.

Метрики считаются на всех 400 честных OOF-запросах. Финальная LR/graph-голова реконструируется отдельно для каждого fold: голова и co-label/`gt_freq` видят только `train(fold)`, а `scores/blend/calibration.npz` намеренно не читается, поскольку это all-dev fit для test-инференса. Один заранее фиксированный fold используется только как дешёвый screening proxy.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 42
PROXY_FOLD = 0          # фиксируем до просмотра perturbation-результатов
EXPORT_VARIANTS = False # read-only по умолчанию

def find_project_root() -> Path:
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / 'candidate_data/calibration.f').exists() and (start / 'solution').exists():
            return start.resolve()
    raise FileNotFoundError('Не найден корень проекта')

ROOT = find_project_root()
SOLUTION = ROOT / 'solution'
EXPERIMENTS = SOLUTION / 'experiments'
if str(SOLUTION / 'src') not in sys.path:
    sys.path.insert(0, str(SOLUTION / 'src'))
print('ROOT:', ROOT)

ROOT: /home/ivan/Desktop/Avito_summer_2026/try_0


/home/ivan/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## 1. Автоматический выбор лучшего завершённого OOF-прогона

Берём только артефакты с 400 dev-запросами и готовым `blend_report.json`. Незавершённый рефит на всех 500 автоматически исключается. Это позволяет ноутбуку подхватить результат ночного sweep без ручной замены пути.

In [2]:
def chosen_blend_map(report_path: Path) -> float:
    report = json.loads(report_path.read_text())
    chosen = report.get('chosen', 'lr')
    retriever = 'blend' if chosen in (None, 'blend') else f'blend_{chosen}'
    rows = [row for row in report['rows'] if row.get('split') == 'dev_oof']
    exact = [row for row in rows if row.get('retriever') == retriever]
    return float((exact or rows)[-1]['map@10'])

artifact_rows = []
for path in sorted(SOLUTION.glob('artifacts*/runs/blend_report.json')):
    artifact = path.parent.parent
    split_path = artifact / 'splits/folds.json'
    required = [split_path, artifact / 'answers/config_snapshot.yaml',
                artifact / 'scores/reranker/calibration.npz',
                artifact / 'scores/fusion/calibration.npz',
                artifact / 'scores/bm25/calibration.npz',
                artifact / 'scores/char_tfidf/calibration.npz',
                artifact / 'scores/dense/calibration.npz',
                artifact / 'scores/dense/index/embeddings.npz']
    if not all(item.exists() for item in required):
        continue
    split = json.loads(split_path.read_text())
    if len(split.get('dev', [])) != 400:
        continue
    artifact_rows.append({'artifact': artifact, 'map@10': chosen_blend_map(path)})

artifact_table = pd.DataFrame(artifact_rows).sort_values('map@10', ascending=False).reset_index(drop=True)
display(artifact_table.assign(artifact=artifact_table['artifact'].map(lambda p: p.name)))
ARTIFACT = Path(artifact_table.iloc[0]['artifact'])
print(f'Выбран: {ARTIFACT.name}, OOF MAP@10={artifact_table.iloc[0]["map@10"]:.4f}')

,artifact,map@10
0,artifacts_rr_ft_k50_ep3,0.7608
1,artifacts_rr_ft_k50,0.7514
2,artifacts_rr_ft_lemma,0.7484
3,artifacts_rr_ft_k30,0.7422
4,artifacts_rr_ft_k50_ep1,0.7361
5,artifacts_rr_bge_ft_labeled,0.7304
6,artifacts_rr_bge_ft_synth,0.5922
7,artifacts_rr_bge_zs_c40,0.5922
8,artifacts,0.5854
9,artifacts_rr_bge_v2_m3,0.5854


Выбран: artifacts_rr_ft_k50_ep3, OOF MAP@10=0.7608


In [3]:
from omegaconf import OmegaConf

from support_search.contracts import ScoreMatrix
from support_search.data.splits import Splits
from support_search.features import GraphFeaturizer, article_vectors, link_adjacency
from support_search.ranking import oof_blend_ltr
from support_search.retrievers.dense import DenseRetriever

articles = pd.read_feather(ROOT / 'candidate_data/articles.f')
calibration = pd.read_feather(ROOT / 'candidate_data/calibration.f')
article_title = articles.set_index('article_id')['title'].fillna('').astype(str).to_dict()
query_text = calibration.set_index('query_id')['query_text'].fillna('').astype(str).to_dict()
ground_truth = {
    int(qid): {int(value) for value in str(raw).split()}
    for qid, raw in zip(calibration['query_id'], calibration['ground_truth'])
}
split_data = json.loads((ARTIFACT / 'splits/folds.json').read_text())
dev_ids = sorted(map(int, split_data['dev']))
holdout_ids = sorted(map(int, split_data['holdout']))
fold_of = {int(qid): int(fold) for qid, fold in split_data['fold_of'].items()}

def load_score_matrix(source: str) -> dict[str, np.ndarray]:
    path = ARTIFACT / f'scores/{source}/calibration.npz'
    with np.load(path, allow_pickle=False) as data:
        return {name: data[name] for name in ('query_ids', 'article_ids', 'scores')}

def rankings(matrix: dict[str, np.ndarray]) -> dict[int, list[int]]:
    result = {}
    article_ids = matrix['article_ids'].astype(int)
    for qid, row in zip(matrix['query_ids'].astype(int), matrix['scores']):
        order = np.lexsort((article_ids, -row))
        result[int(qid)] = article_ids[order].tolist()
    return result

def ap_at_10(predicted: list[int], relevant: set[int]) -> float:
    hits, value = 0, 0.0
    for rank, aid in enumerate(predicted[:10], start=1):
        if aid in relevant:
            hits += 1
            value += hits / rank
    return value / min(len(relevant), 10) if relevant else 0.0

def recall_at_k(predicted: list[int], relevant: set[int], k: int) -> float:
    return len(set(predicted[:k]) & relevant) / len(relevant) if relevant else 0.0

def first_relevant_rank(predicted: list[int], relevant: set[int]) -> int:
    return next((rank for rank, aid in enumerate(predicted, start=1) if aid in relevant), 999)

# ВАЖНО: scores/blend/calibration.npz — all-dev fit. Для dev-диагностики он запрещён.
# Восстанавливаем ровно тот же lr_graph OOF, что stage_blend: пять независимых
# голов, а co-label/gt_freq каждого fold строятся только по train(fold).
snapshot = OmegaConf.load(ARTIFACT / 'answers/config_snapshot.yaml')
blend_report = json.loads((ARTIFACT / 'runs/blend_report.json').read_text())
assert blend_report.get('chosen') == 'lr_graph', (
    'Ноутбук требует явного OOF-реконструктора для выбранной головы', blend_report.get('chosen')
)
fusion_report = json.loads((ARTIFACT / 'runs/fusion_report.json').read_text())
ltr_sources = list(fusion_report['sources'])
source_matrices = {
    name: ScoreMatrix.load(ARTIFACT / f'scores/{name}/calibration.npz')
    for name in ltr_sources
}
reranker_matrix = ScoreMatrix.load(ARTIFACT / 'scores/reranker/calibration.npz')
splits = Splits.from_json(split_data)

dev_set, holdout_set = set(dev_ids), set(holdout_ids)
for fold in range(splits.n_splits):
    train_set, val_set = set(splits.train_ids(fold)), set(splits.fold(fold))
    assert train_set.isdisjoint(val_set)
    assert holdout_set.isdisjoint(train_set | val_set)
    assert train_set | val_set == dev_set

body_of = articles.set_index('article_id')['body'].fillna('').astype(str).to_dict()
article_ids = reranker_matrix.article_ids.astype(int)
link_adj = link_adjacency(article_ids, [body_of[int(aid)] for aid in article_ids])
dense_index = DenseRetriever.load(ARTIFACT / 'scores/dense/index', encoder=None)
dense_vectors = article_vectors(
    dense_index.chunk_emb, dense_index.title_emb, dense_index.chunk_offsets
)
dense_row = {int(aid): i for i, aid in enumerate(dense_index.article_ids)}
dense_vectors = dense_vectors[[dense_row[int(aid)] for aid in article_ids]]
graph_cfg = snapshot.ranking.graph_features
graph_featurizer = GraphFeaturizer(
    article_ids, link_adj=link_adj, article_vec=dense_vectors,
    top_m=int(graph_cfg.top_m), interaction=bool(graph_cfg.interaction),
    co_weight=str(graph_cfg.co_weight),
)
oof_blend_rankings, oof_feature_names = oof_blend_ltr(
    source_matrices, reranker_matrix, ground_truth, splits, names=ltr_sources,
    l2=float(snapshot.ranking.ltr.l2),
    use_ranks=bool(snapshot.ranking.ltr.use_ranks),
    depth=len(article_ids), featurizer=graph_featurizer,
)
assert set(oof_blend_rankings) == dev_set

expected_oof = float(next(
    row['map@10'] for row in blend_report['rows']
    if row.get('split') == 'dev_oof' and row.get('retriever') == 'blend_lr_graph'
))
actual_oof = float(np.mean([ap_at_10(oof_blend_rankings[q], ground_truth[q]) for q in dev_ids]))
assert abs(actual_oof - expected_oof) < 5e-5, (actual_oof, expected_oof)

sources = [name for name in ('bm25', 'char_tfidf', 'dense', 'fusion', 'reranker')
           if (ARTIFACT / f'scores/{name}/calibration.npz').exists()] + ['blend']
rank_by_source = {name: rankings(load_score_matrix(name)) for name in sources if name != 'blend'}
rank_by_source['blend'] = oof_blend_rankings

# Числовой audit trail: сохранённые dev-строки fusion/reranker обязаны совпасть
# со своими OOF-манифестами; blend обязан совпасть с fold-wise отчётом выше.
stage_expected = {
    'fusion': float(json.loads((ARTIFACT / 'scores/fusion/manifest.json').read_text())['oof_map']),
    'reranker': float(json.loads((ARTIFACT / 'scores/reranker/manifest.json').read_text())['fine_tune_map']),
    'blend': expected_oof,
}
for source, expected in stage_expected.items():
    observed = float(np.mean([ap_at_10(rank_by_source[source][q], ground_truth[q]) for q in dev_ids]))
    assert abs(observed - expected) < 5e-5, (source, observed, expected)

print(f'dev={len(dev_ids)}, holdout={len(holdout_ids)}, sources={sources}')
print(f'Leak-free audit OK: fold-wise lr_graph OOF MAP@10={actual_oof:.6f}; all-dev blend не загружен')

/home/ivan/.local/lib/python3.12/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


dev=400, holdout=100, sources=['bm25', 'char_tfidf', 'dense', 'fusion', 'reranker', 'blend']
Leak-free audit OK: fold-wise lr_graph OOF MAP@10=0.760759; all-dev blend не загружен


## 2. Воронка качества и полные ошибки

Полный промах важнее среднего MAP: сначала проверяем, был ли хотя бы один GT в top-50 fusion, затем — поднял ли финал хотя бы один GT в top-10. `fusion` и `reranker` содержат OOF-строки dev по контракту стадий; `blend` ниже заново построен fold-wise, включая граф только из `train(fold)`.

In [4]:
rows = []
for qid in dev_ids:
    row = {'query_id': qid, 'query_text': query_text[qid], 'n_gt': len(ground_truth[qid]),
           'fold': fold_of[qid]}
    for source in sources:
        predicted = rank_by_source[source][qid]
        row[f'ap_{source}'] = ap_at_10(predicted, ground_truth[qid])
        row[f'recall10_{source}'] = recall_at_k(predicted, ground_truth[qid], 10)
        row[f'first_{source}'] = first_relevant_rank(predicted, ground_truth[qid])
    rows.append(row)
diagnostics = pd.DataFrame(rows)

summary_rows = []
for source in sources:
    summary_rows.append({
        'stage': source,
        'MAP@10': diagnostics[f'ap_{source}'].mean(),
        'hit@10': (diagnostics[f'first_{source}'] <= 10).mean(),
        'mean first rank (cap=51)': diagnostics[f'first_{source}'].clip(upper=51).mean(),
    })
display(pd.DataFrame(summary_rows).style.format({
    'MAP@10': '{:.4f}', 'hit@10': '{:.1%}', 'mean first rank (cap=51)': '{:.2f}'
}))

candidate_miss = diagnostics['first_fusion'] > 50
ranker_miss = (diagnostics['first_fusion'] <= 50) & (diagnostics['first_blend'] > 10)
partial = (diagnostics['first_blend'] <= 10) & (diagnostics['ap_blend'] < 1)
funnel = pd.DataFrame({
    'group': ['нет GT в fusion top-50', 'GT есть в top-50, но нет в final top-10',
              'есть hit@10, но AP<1', 'идеальный AP=1'],
    'n': [candidate_miss.sum(), ranker_miss.sum(), partial.sum(), (diagnostics['ap_blend'] == 1).sum()],
})
display(funnel)

,stage,MAP@10,hit@10,mean first rank (cap=51)
0,bm25,0.3176,71.2%,11.46
1,char_tfidf,0.2237,57.8%,15.12
2,dense,0.4177,82.5%,7.00
3,fusion,0.5058,89.8%,4.68
4,reranker,0.6894,96.5%,2.75
5,blend,0.7608,98.0%,2.28


,group,n
0,нет GT в fusion top-50,4
1,"GT есть в top-50, но нет в final top-10",4
2,"есть hit@10, но AP<1",185
3,идеальный AP=1,207


In [5]:
def titles(ids) -> str:
    return ' | '.join(f'{aid}: {article_title.get(aid, "")}' for aid in ids)

hard_error_rows = []
for item in diagnostics[candidate_miss | ranker_miss].itertuples():
    qid = int(item.query_id)
    hard_error_rows.append({
        'query_id': qid,
        'failure_stage': 'candidate' if item.first_fusion > 50 else 'ranking',
        'fusion / reranker / final rank': f'{item.first_fusion} / {item.first_reranker} / {item.first_blend}',
        'query': query_text[qid],
        'GT': titles(sorted(ground_truth[qid])),
        'final top-3': titles(rank_by_source['blend'][qid][:3]),
    })
hard_errors = pd.DataFrame(hard_error_rows)
pd.set_option('display.max_colwidth', 180)
display(hard_errors)

,query_id,failure_stage,fusion / reranker / final rank,query,GT,final top-3
0,70,candidate,144 / 104 / 104,"Здравствуйте хотел бы узнать, почему такая сумма большая если носки стоят <MONEY>",1951: Кто оплачивает доставку и сколько она стоит,4384: Баланс для покупок | 4395: Бонусы | 3128: Почему моё объявление платное?
1,73,ranking,14 / 12 / 13,"У меня в авто теки висит ограничения на регистрацию , на автомобиль хендай салярис гос номер <ID>",4318: Как поддерживать достоверность описаний в Авто,4232: Автотека | 4321: Мой уровень сервиса | 4255: Подписка «Авито × Хараба» для профессиональных продавцов авто
2,254,candidate,121 / 549 / 549,"Заказ <ID> Скажите, когда мне вернутся деньги?",4218: Тариф с оплатой за просмотры в Товарах,2865: Когда вернутся деньги за доставку | 4219: Покупателю | 2646: Оплата заказов с доставкой
3,256,ranking,3 / 43 / 26,"Добрый день. Подскажите пожалуйста, а где мои 10тысяч которые я вывел на т-банк себе?",2943: Не могу вывести деньги за доставку,"4384: Баланс для покупок | 4219: Покупателю | 4313: Баланс для услуг Авито (объявления, продвижение)"
4,272,ranking,15 / 11 / 11,Авито доставку могу в любой сдэк сдавать ?,1909: Отправить заказ,4286: Грузовая доставка | 4328: Что можно заказать и отправить | 4234: Как продавать и покупать с доставкой
5,360,candidate,95 / 546 / 546,Почему доставка покупателям за мой счет????? Отключите эту функцию алло!!!!,"4214: Скидки, бонусы и промокоды",4234: Как продавать и покупать с доставкой | 4362: Включить или отключить доставку | 1951: Кто оплачивает доставку и сколько она стоит
6,418,ranking,36 / 22 / 33,"О каких накрутках речь, я заказал духи в другом городе, авито доставкой?!",3261: Отзыв отклонили,"1951: Кто оплачивает доставку и сколько она стоит | 4214: Скидки, бонусы и промокоды | 2698: Как продавцу подключить скидки"
7,453,candidate,254 / 507 / 507,Эффективный Черчилль: Дмитрий Медведев Здравствуйте! Эта книга разрешена к продаже на площадке?,4133: Ограничены звонки и чаты с покупателями,"4162: В каких категориях действуют изменённые условия продажи | 4243: Вещи, техника и другие товары | 4436: Уровень сервиса"


## 3. Свойства запросов и отрицательный перенос

Это описательные срезы, а не причинные эксперименты. Они помогают выбрать запросы для ручного разбора: длинные, с плейсхолдерами, с неизвестными словарю словами, с несколькими GT и случаи, где FT ухудшил fusion.

In [6]:
CYRILLIC_WORD = re.compile(r'[а-яё]{4,}', re.IGNORECASE)
PLACEHOLDER = re.compile(r'<(?:MONEY|DATE|PHONE|URL|ID)>')
DOMAIN_WORDS = {'авито', 'авитодоставка', 'авитодоставкой', 'сдек', 'боксберри',
                'постамат', 'постамате', 'яндекс', 'аккаунт', 'айди'}

try:
    import pymorphy3
    import pymorphy3_dicts_ru
    morph = pymorphy3.MorphAnalyzer(path=pymorphy3_dicts_ru.get_path())
    def unknown_tokens(text: str) -> list[str]:
        words = {word.lower() for word in CYRILLIC_WORD.findall(text)}
        return sorted(word for word in words if word not in DOMAIN_WORDS and not morph.parse(word)[0].is_known)
except ImportError:
    morph = None
    def unknown_tokens(text: str) -> list[str]:
        return []

diagnostics['n_words'] = diagnostics['query_text'].str.findall(r'\w+').str.len()
diagnostics['n_chars'] = diagnostics['query_text'].str.len()
diagnostics['has_placeholder'] = diagnostics['query_text'].str.contains(PLACEHOLDER)
diagnostics['unknown_tokens'] = diagnostics['query_text'].map(unknown_tokens)
diagnostics['has_unknown'] = diagnostics['unknown_tokens'].str.len() > 0
diagnostics['length_bin'] = pd.cut(diagnostics['n_words'], [0, 7, 12, 20, np.inf],
                                   labels=['1–7', '8–12', '13–20', '21+'])
diagnostics['delta_ft_vs_fusion'] = diagnostics['ap_reranker'] - diagnostics['ap_fusion']
diagnostics['delta_blend_vs_ft'] = diagnostics['ap_blend'] - diagnostics['ap_reranker']

def grouped_quality(column: str) -> pd.DataFrame:
    return diagnostics.groupby(column, observed=True).agg(
        n=('query_id', 'size'), map_final=('ap_blend', 'mean'),
        candidate_miss=('first_fusion', lambda x: (x > 50).mean()),
        top10_miss=('first_blend', lambda x: (x > 10).mean()),
        delta_ft=('delta_ft_vs_fusion', 'mean'),
    ).reset_index()

for column in ('length_bin', 'has_placeholder', 'has_unknown', 'n_gt'):
    print('Срез:', column)
    display(grouped_quality(column).style.format({
        'map_final': '{:.4f}', 'candidate_miss': '{:.1%}', 'top10_miss': '{:.1%}', 'delta_ft': '{:+.4f}'
    }))

Срез: length_bin


,length_bin,n,map_final,candidate_miss,top10_miss,delta_ft
0,1–7,107,0.7868,0.9%,1.9%,+0.1942
1,8–12,144,0.7433,2.1%,2.8%,+0.1803
2,13–20,115,0.7579,0.0%,1.7%,+0.2124
3,21+,34,0.7623,0.0%,0.0%,+0.0668


Срез: has_placeholder


,has_placeholder,n,map_final,candidate_miss,top10_miss,delta_ft
0,False,360,0.7631,0.6%,1.4%,+0.1741
1,True,40,0.7394,5.0%,7.5%,+0.2690


Срез: has_unknown


,has_unknown,n,map_final,candidate_miss,top10_miss,delta_ft
0,False,333,0.7635,1.2%,1.8%,+0.2043
1,True,67,0.7473,0.0%,3.0%,+0.0807


Срез: n_gt


,n_gt,n,map_final,candidate_miss,top10_miss,delta_ft
0,1,223,0.7548,1.8%,3.6%,+0.1813
1,2,146,0.7735,0.0%,0.0%,+0.1636
2,3,30,0.7387,0.0%,0.0%,+0.2806
3,4,1,0.8875,0.0%,0.0%,+0.7125


In [7]:
regressions = diagnostics.nsmallest(20, 'delta_ft_vs_fusion').copy()
regressions['GT'] = regressions['query_id'].map(lambda q: titles(sorted(ground_truth[q])))
regressions['fusion top-1'] = regressions['query_id'].map(lambda q: titles(rank_by_source['fusion'][q][:1]))
regressions['FT top-1'] = regressions['query_id'].map(lambda q: titles(rank_by_source['reranker'][q][:1]))
display(regressions[[
    'query_id', 'delta_ft_vs_fusion', 'ap_fusion', 'ap_reranker', 'ap_blend',
    'query_text', 'unknown_tokens', 'GT', 'fusion top-1', 'FT top-1'
]].style.format({
    'delta_ft_vs_fusion': '{:+.3f}', 'ap_fusion': '{:.3f}', 'ap_reranker': '{:.3f}', 'ap_blend': '{:.3f}'
}))

,query_id,delta_ft_vs_fusion,ap_fusion,ap_reranker,ap_blend,query_text,unknown_tokens,GT,fusion top-1,FT top-1
23,28,-1.000,1.000,0.000,0.500,"Добрый день! Подскажите, не вижу объект в черновиках у сотрудника (Галина Светлая). В чем может быть проблема?",[],2661: Не вижу объявление в профиле,2661: Не вижу объявление в профиле,4292: Резюме
235,287,-1.000,1.000,0.000,1.000,"за последние 3 дня у меня один заказ, хотя просмотры на том же уровне. может быть я в каком то теневом бане, такого не было у меня",[],2948: Почему мало просмотров,2948: Почему мало просмотров,4229: Статистика для профессиональных продавцов
260,319,-0.900,1.000,0.100,0.250,"Здравствуйте!Прошло больше чем 5 дней,где мои деньги ?",[],1966: Когда вернутся деньги после отмены доставки в пункт выдачи,1966: Когда вернутся деньги после отмены доставки в пункт выдачи,4361: Продавцу
382,478,-0.900,1.000,0.100,0.333,Здравствуйте хочу купить фару на авто но при оформлении нет сдек доставки,[],4308: Заказать товар с доставкой,4308: Заказать товар с доставкой,4219: Покупателю
65,76,-0.857,1.000,0.143,0.333,Здравствуйте! Почему при заказе доступна только Яндекс доставка?,[],4396: Не могу отправить заказ,4396: Не могу отправить заказ,4328: Что можно заказать и отправить
309,383,-0.833,1.000,0.167,0.500,"Здравствуйте! Заказ завис с авито доставкой, денги сняли, товар продовец не выслал...","['денги', 'продовец']",1958: Продавец не отправляет заказ,1958: Продавец не отправляет заказ,4219: Покупателю
393,493,-0.829,1.000,0.171,0.292,"Если получатель откажется от товара, то за доставку кто будет оплачивать?",[],4234: Как продавать и покупать с доставкой | 4403: Продавцу — товар не забирают или вернули,4403: Продавцу — товар не забирают или вернули,4400: Покупателю — отказаться от товара или вернуть его
94,109,-0.817,1.000,0.183,1.000,"Здравствуйте, отказалась от покупки , заказывала через авитодоставку, деньги до сих пор не вернулись. Как вернуть?",['авитодоставку'],1966: Когда вернутся деньги после отмены доставки в пункт выдачи | 4400: Покупателю — отказаться от товара или вернуть его,4400: Покупателю — отказаться от товара или вернуть его,4384: Баланс для покупок
192,234,-0.800,1.000,0.200,0.333,"Подскажите пожалуйста , я правильно понимаю, что я получу за сумку не ту сумму которую я заявила ? С меня удержат за отправку и продажу ?",[],4361: Продавцу,4361: Продавцу,1951: Кто оплачивает доставку и сколько она стоит
362,451,-0.800,1.000,0.200,0.500,"Здравствуйте, не могу оплатить заказ , пишет заказ оплачен .",[],"4440: Платёж прошёл, но есть проблема","4440: Платёж прошёл, но есть проблема",4219: Покупателю


## 4. Известность статьи относительно train-разметки

Идея уже частично реализована: графовая голова содержит `gt_freq = log1p(частота статьи в GT)`. В текущем отчёте её положительный коэффициент показывает, что prior действительно используется. Но добавлять только бинарное `known/unseen` рискованно: оно усиливает calibration-head и может ещё сильнее ухудшить cold-article generalization.

Ниже известность считается строго по train-части фолда соответствующего запроса. Ноль означает, что модель фолда не видела GT-статью этого запроса как позитив.

In [8]:
dev_frequency = Counter(aid for qid in dev_ids for aid in ground_truth[qid])
fold_frequency = {fold: Counter() for fold in sorted(set(fold_of.values()))}
for qid in dev_ids:
    fold_frequency[fold_of[qid]].update(ground_truth[qid])

knownness_rows = []
for qid in dev_ids:
    fold = fold_of[qid]
    train_freqs = [dev_frequency[aid] - fold_frequency[fold][aid] for aid in ground_truth[qid]]
    unseen = sum(freq == 0 for freq in train_freqs)
    novelty = ('все GT seen' if unseen == 0 else
               'все GT unseen' if unseen == len(train_freqs) else 'часть GT unseen')
    knownness_rows.append({
        'query_id': qid, 'novelty': novelty, 'freq_min': min(train_freqs),
        'freq_max': max(train_freqs), 'freq_sum': sum(train_freqs),
    })
knownness = pd.DataFrame(knownness_rows)
diagnostics = diagnostics.merge(knownness, on='query_id', how='left')
knownness_summary = diagnostics.groupby('novelty', observed=True).agg(
    n=('query_id', 'size'), fusion=('ap_fusion', 'mean'), reranker=('ap_reranker', 'mean'),
    final=('ap_blend', 'mean'), ft_delta=('delta_ft_vs_fusion', 'mean'),
    final_miss=('first_blend', lambda x: (x > 10).mean()),
).reset_index()
display(knownness_summary.style.format({
    'fusion': '{:.3f}', 'reranker': '{:.3f}', 'final': '{:.3f}',
    'ft_delta': '{:+.3f}', 'final_miss': '{:.1%}'
}))

report = json.loads((ARTIFACT / 'runs/blend_report.json').read_text())
coef = report.get('coefficients', {})
print('Текущий коэффициент gt_freq:', coef.get('gt_freq', 'нет в этой голове'))
display(pd.DataFrame([{
    'current feature': 'log1p(train GT frequency)',
    'possible ablation': 'binary unseen + clipped log-frequency + interactions',
    'важный срез': 'all-GT-unseen и статьи с freq 1–2',
    'риск': 'рост общей OOF за счёт head при ухудшении cold-tail',
}]))

,novelty,n,fusion,reranker,final,ft_delta,final_miss
0,все GT seen,367,0.506,0.716,0.784,+0.210,0.8%
1,все GT unseen,25,0.504,0.358,0.484,-0.146,20.0%
2,часть GT unseen,8,0.510,0.493,0.572,-0.018,0.0%


Текущий коэффициент gt_freq: 0.3496


,current feature,possible ablation,важный срез,риск
0,log1p(train GT frequency),binary unseen + clipped log-frequency + interactions,all-GT-unseen и статьи с freq 1–2,рост общей OOF за счёт head при ухудшении cold-tail


### Что стоит проверить для knownness

Наиболее осмысленная версия — не одна бинарная фича, а маленький набор:

- `is_unseen = 1[freq == 0]`;
- `log_freq = log1p(freq)` с clipping, чтобы статья-хаб не доминировала;
- `is_rare = 1[freq <= 2]`;
- взаимодействия `is_unseen × dense_score` и `log_freq × reranker_margin`.

Гипотеза: известным статьям можно доверять popularity-prior, а unseen-кандидата разрешать поднимать только при сильном semantic score. Принимать эксперимент следует не только по общему MAP, но и по двум ограничениям: cold-slice не ухудшается и число уникальных top-1 на test не схлопывается.

Для чистой ablation нужно дать `GraphFeaturizer` возможность включать/выключать `gt_freq` отдельно. Сейчас она входит в пакет графовых фич, поэтому из готового `blend_report` её индивидуальный вклад не идентифицируется. Это потребует небольшой правки вне `experiments/`; в этом ноутбуке она намеренно не сделана.

## 5. Можно ли использовать только одну из пяти моделей?

Да — как screening. Модель `fold_0` обучена на 4/5 dev и честно предсказывает свой отложенный `fold_0` (~80 запросов). Это сокращает число query–passage пар примерно в пять раз. Но единичный fold нельзя использовать для финального вывода: редкие статьи, опечатки и полные промахи распределены между фолдами неравномерно.

Ниже по уже готовым OOF-скорам показано, насколько гуляют абсолютные метрики и paired-эффект между фолдами.

In [9]:
fold_rows = []
for fold in sorted(set(fold_of.values())):
    part = diagnostics[diagnostics['fold'] == fold]
    fold_rows.append({
        'fold': fold, 'n': len(part), 'fusion': part['ap_fusion'].mean(),
        'reranker': part['ap_reranker'].mean(), 'final': part['ap_blend'].mean(),
        'FT − fusion': (part['ap_reranker'] - part['ap_fusion']).mean(),
        'final − FT': (part['ap_blend'] - part['ap_reranker']).mean(),
        'hard misses': (part['first_blend'] > 10).sum(),
    })
fold_table = pd.DataFrame(fold_rows)
display(fold_table.style.format({
    'fusion': '{:.4f}', 'reranker': '{:.4f}', 'final': '{:.4f}',
    'FT − fusion': '{:+.4f}', 'final − FT': '{:+.4f}'
}))

def paired_bootstrap_ci(values, n_resamples=10_000, seed=RANDOM_SEED):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(n_resamples, len(values)))].mean(axis=1)
    return values.mean(), *np.quantile(means, [0.025, 0.975])

proxy = diagnostics[diagnostics['fold'] == PROXY_FOLD]
all_effect = diagnostics['ap_blend'] - diagnostics['ap_reranker']
proxy_effect = proxy['ap_blend'] - proxy['ap_reranker']
reliability = pd.DataFrame([
    {'sample': 'all 5 folds', 'n': len(all_effect), 'mean / low / high': paired_bootstrap_ci(all_effect)},
    {'sample': f'only fold {PROXY_FOLD}', 'n': len(proxy_effect), 'mean / low / high': paired_bootstrap_ci(proxy_effect)},
])
display(reliability)
print('Compute ratio by number of validation queries:', len(diagnostics) / len(proxy))

,fold,n,fusion,reranker,final,FT − fusion,final − FT,hard misses
0,0,82,0.5515,0.6855,0.7490,+0.1340,+0.0635,1
1,1,80,0.5043,0.6815,0.7567,+0.1772,+0.0752,0
2,2,80,0.5200,0.7031,0.7672,+0.1831,+0.0641,2
3,3,79,0.4785,0.7006,0.7630,+0.2221,+0.0624,3
4,4,79,0.4728,0.6766,0.7683,+0.2037,+0.0918,2


,sample,n,mean / low / high
0,all 5 folds,400,"(0.07132390873015874, 0.05091326884920635, 0.09198188244047617)"
1,only fold 0,82,"(0.06349287004774809, 0.01997836414376047, 0.11025303668215251)"


Compute ratio by number of validation queries: 4.878048780487805


### Рекомендуемый двухступенчатый протокол

1. **Screen:** заранее фиксируем `fold_0`, кандидатов и варианты. Одна fold-модель оценивает только свои ~80 validation-запросов — без test и без остальных четырёх моделей.
2. Отбрасываем вариант, если paired-эффект отрицателен или практически нулевой.
3. **Confirm:** только победителя считаем на всех пяти OOF-фолдах.

При top-50 и трёх чанках один вариант на одном fold — около `80 × 50 × 3 = 12 000` пар. При наблюдавшихся ~64 парах/с это примерно 3–4 минуты чистого inference; восемь вариантов — около 25–35 минут. Полное подтверждение — примерно в пять раз дороже.

Важно: штатный `cli all` после каждого fold дополнительно скорит 500 test-запросов и уничтожает эту экономию. Для proxy нужен маленький внешний runner, который получает только `fold_0` и не трогает test.

## 6. Контрфактические преобразования запросов

Набор разделён по проверяемой причине, а не по красоте аугментации:

| Вариант | Что проверяет | Ожидание |
|---|---|---|
| `case_punct` | регистр и пунктуационный шум | почти полная инвариантность |
| `typo_delete` | пропуск одной внутренней буквы | малая деградация |
| `typo_swap` | перестановка соседних букв | малая деградация |
| `typo_keyboard` | соседняя клавиша русской раскладки | малая деградация |
| `typo_two` | две ошибки в разных словах | контролируемая более сильная деградация |
| `drop_content` | зависимость от одного смыслового токена | выявляет хрупкие запросы |
| `polite_noise` | лишнее начало обращения | инвариантность к разговорной обвязке |
| `topic_distractor` | конкурирующий частый intent | склонность цепляться за delivery/payment prior |
| `mask_entities` | цифры, даты, ID и суммы | реальная ценность сущностей/плейсхолдеров |

Варианты детерминированы по `(query_id, variant)` и не меняют GT. Для GPU-screening генерируется только заранее выбранный fold.

In [10]:
RUSSIAN_STOP = {
    'здравствуйте', 'подскажите', 'пожалуйста', 'добрый', 'день', 'авито',
    'когда', 'почему', 'какой', 'какая', 'какие', 'через', 'можно', 'меня',
    'мне', 'если', 'этот', 'этого', 'который', 'товар', 'заказ',
}
KEYBOARD_NEIGHBOR = {
    'й':'ц', 'ц':'у', 'у':'к', 'к':'е', 'е':'н', 'н':'г', 'г':'ш', 'ш':'щ', 'щ':'з', 'з':'х', 'х':'ъ',
    'ф':'ы', 'ы':'в', 'в':'а', 'а':'п', 'п':'р', 'р':'о', 'о':'л', 'л':'д', 'д':'ж', 'ж':'э',
    'я':'ч', 'ч':'с', 'с':'м', 'м':'и', 'и':'т', 'т':'ь', 'ь':'б', 'б':'ю',
}
WORD_PATTERN = re.compile(r'[А-Яа-яЁё]{5,}')

def stable_rng(qid: int, variant: str) -> np.random.Generator:
    raw = hashlib.sha256(f'{RANDOM_SEED}:{qid}:{variant}'.encode()).digest()[:8]
    return np.random.default_rng(int.from_bytes(raw, 'little'))

def content_matches(text: str):
    return [match for match in WORD_PATTERN.finditer(text)
            if match.group(0).lower() not in RUSSIAN_STOP]

def replace_match(text: str, match: re.Match, replacement: str) -> str:
    return text[:match.start()] + replacement + text[match.end():]

def mutate_word(word: str, kind: str, rng: np.random.Generator) -> str:
    chars = list(word)
    positions = list(range(1, max(2, len(chars) - 1)))
    pos = int(rng.choice(positions))
    if kind == 'delete':
        del chars[pos]
    elif kind == 'swap' and pos + 1 < len(chars):
        chars[pos], chars[pos + 1] = chars[pos + 1], chars[pos]
    elif kind == 'keyboard':
        source = chars[pos].lower()
        replacement = KEYBOARD_NEIGHBOR.get(source, source)
        chars[pos] = replacement.upper() if chars[pos].isupper() else replacement
    return ''.join(chars)

def mutate_one(text: str, qid: int, variant: str, kind: str) -> str:
    matches = content_matches(text)
    if not matches:
        return text
    rng = stable_rng(qid, variant)
    match = matches[int(rng.integers(0, len(matches)))]
    return replace_match(text, match, mutate_word(match.group(0), kind, rng))

def make_variant(text: str, qid: int, variant: str) -> str:
    if variant == 'original':
        return text
    if variant == 'case_punct':
        return '!!! ' + text.swapcase() + ' ???'
    if variant == 'typo_delete':
        return mutate_one(text, qid, variant, 'delete')
    if variant == 'typo_swap':
        return mutate_one(text, qid, variant, 'swap')
    if variant == 'typo_keyboard':
        return mutate_one(text, qid, variant, 'keyboard')
    if variant == 'typo_two':
        first = mutate_one(text, qid, variant + ':1', 'delete')
        return mutate_one(first, qid, variant + ':2', 'keyboard')
    if variant == 'drop_content':
        matches = content_matches(text)
        if not matches:
            return text
        rng = stable_rng(qid, variant)
        match = matches[int(rng.integers(0, len(matches)))]
        return re.sub(r'\s+', ' ', replace_match(text, match, '')).strip()
    if variant == 'polite_noise':
        return 'Здравствуйте! Подскажите, пожалуйста, очень срочно: ' + text
    if variant == 'topic_distractor':
        return text + ' Также не понимаю, почему не работает Авито Доставка и когда вернут деньги.'
    if variant == 'mask_entities':
        masked = re.sub(r'<(?:MONEY|DATE|PHONE|URL|ID)>', '<ENTITY>', text)
        return re.sub(r'\d+(?:[.,]\d+)?', '<NUMBER>', masked)
    raise KeyError(variant)

variant_names = [
    'original', 'case_punct', 'typo_delete', 'typo_swap', 'typo_keyboard',
    'typo_two', 'drop_content', 'polite_noise', 'topic_distractor', 'mask_entities',
]
proxy_ids = sorted(qid for qid in dev_ids if fold_of[qid] == PROXY_FOLD)
variant_rows = []
for qid in proxy_ids:
    for variant in variant_names:
        variant_rows.append({
            'query_id': qid, 'fold': PROXY_FOLD, 'variant': variant,
            'query_text': make_variant(query_text[qid], qid, variant),
            'original_text': query_text[qid],
            'ground_truth': ' '.join(map(str, sorted(ground_truth[qid]))),
        })
variants = pd.DataFrame(variant_rows)
display(variants.groupby('variant').size().rename('n').to_frame())
display(variants[variants['query_id'].isin(proxy_ids[:3])][['query_id', 'variant', 'query_text']])

,n
variant,
case_punct,82
drop_content,82
mask_entities,82
original,82
polite_noise,82
topic_distractor,82
typo_delete,82
typo_keyboard,82
typo_swap,82


,query_id,variant,query_text
0,3,original,Здравствуйте. Как отправить товар через Авито.
1,3,case_punct,!!! зДРАВСТВУЙТЕ. кАК ОТПРАВИТЬ ТОВАР ЧЕРЕЗ аВИТО. ???
2,3,typo_delete,Здравствуйте. Как отправиь товар через Авито.
3,3,typo_swap,Здравствуйте. Как отпарвить товар через Авито.
4,3,typo_keyboard,Здравствуйте. Как отправиьь товар через Авито.
5,3,typo_two,Здравствуйте. Как отпавиьь товар через Авито.
6,3,drop_content,Здравствуйте. Как товар через Авито.
7,3,polite_noise,"Здравствуйте! Подскажите, пожалуйста, очень срочно: Здравствуйте. Как отправить товар через Авито."
8,3,topic_distractor,"Здравствуйте. Как отправить товар через Авито. Также не понимаю, почему не работает Авито Доставка и когда вернут деньги."
9,3,mask_entities,Здравствуйте. Как отправить товар через Авито.


### Естественные опечатки: ручной paired-набор

Автоматический словарь смешивает опечатки с брендами (`СДЭК`, модели автомобилей) и жаргоном. Поэтому он используется только для отбора кандидатов. В таблице ниже нужно заполнить две колонки:

- `is_real_typo`: действительно ли это опечатка;
- `corrected_text`: минимально исправленный текст без стилистического переписывания.

Сравнение `original ↔ corrected` будет самым прямым причинным тестом естественных опечаток.

In [11]:
natural_typo_candidates = diagnostics[diagnostics['has_unknown']].copy()
natural_typo_candidates['is_real_typo'] = ''
natural_typo_candidates['corrected_text'] = ''
natural_typo_review = natural_typo_candidates[[
    'query_id', 'fold', 'unknown_tokens', 'query_text', 'ap_fusion', 'ap_reranker', 'ap_blend',
    'is_real_typo', 'corrected_text'
]]
display(natural_typo_review)

if EXPORT_VARIANTS:
    generated = EXPERIMENTS / 'generated'
    generated.mkdir(exist_ok=True)
    variants.to_csv(generated / f'query_variants_fold{PROXY_FOLD}.csv', index=False)
    natural_typo_review.to_csv(generated / 'natural_typo_review.csv', index=False)
    print('Экспортировано в', generated)
else:
    print('Read-only режим: для экспорта установите EXPORT_VARIANTS=True и перезапустите эту ячейку.')

,query_id,fold,unknown_tokens,query_text,ap_fusion,ap_reranker,ap_blend,is_real_typo,corrected_text
2,4,2,[возрат],как получить деньги за возрат если продавец уже забрал товар?,1.000000,0.236111,0.392857,,
3,5,3,[прийдут],"Когда мне прийдут деньги за доставку, сегодня уже <DATE>",0.250000,1.000000,1.000000,,
26,31,2,[обявлению],"Здравствуйте, у меня по обявлению при цене <MONEY>, доставка в 5пост стоит <MONEY>. Что сделать, чтобы была адекватная цена доставки, подскажите пожалуйста!",0.500000,1.000000,0.642857,,
35,41,3,[поишли],"Прошу вернуть деньги.Я вам отправила чек.Вы обещали вернуть за невыполнение заказа вернуть деньги.,деньги не поишли",0.000000,0.625000,0.600000,,
41,48,4,[небели],"когда вернут деньги, возврат уже выдан продавцу, я заказывала через авито доставку, должны были вернуть деньги в течении 1-5 дней, прошло 2 небели",0.166667,0.583333,0.700000,,
...,...,...,...,...,...,...,...,...,...
383,479,4,[бали],"Подскажите, я хочу выставить объявление по услугам риелтора на бали, в какую категорию мне лучше выставлять его?",0.333333,0.333333,0.500000,,
388,484,4,[красноперекопска],"Здравствуйте скажите пожалуйста а можно из Крыма Красноперекопска отправить посылку коляску для новорождённых, габариты высота 110 см ширина 60 на 60 см?",0.500000,1.000000,1.000000,,
392,492,2,[тинькофф],"Почему я не могу оплатить товар с доставкой? Идет очень долгая загрузка,оплата СПБ,тинькофф и кошельком авито. В чем проблема????",0.700000,0.583333,1.000000,,
394,495,0,[безопастно],Как отправить безопастно ювелирное изделие?,0.250000,0.200000,0.250000,,


Read-only режим: для экспорта установите EXPORT_VARIANTS=True и перезапустите эту ячейку.


## 7. GPU-screening после завершения текущего обучения

Ноутбук не должен запускать GPU. Нужен отдельный маленький runner со следующим контрактом:

**Входы**

- `experiments/generated/query_variants_fold0.csv`;
- кандидаты только соответствующих query ID из `<artifact>/candidates/dev_oof.parquet`;
- passages/chunks из выбранного artifact;
- одна модель `<artifact>/models/reranker_ft/fold_0`.

**Режим A — fixed candidates, приоритетный:** меняется только текст запроса, кандидаты остаются исходными. Он изолирует устойчивость cross-encoder и стоит дёшево.

**Режим B — full query path:** заново считаются BM25/char/dense/fusion для варианта, затем cross-encoder. Он показывает суммарную устойчивость всего pipeline, но нужен только для вариантов, прошедших режим A.

**Выход** — parquet с колонками `query_id, variant, article_id, score`. По 50 строк на `(query_id, variant)`.

Предлагаемая команда после добавления runner внутри `solution/experiments/`:

```bash
cd solution
CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src HF_HUB_OFFLINE=1 \
python3 experiments/score_query_variants.py \
  --variants experiments/generated/query_variants_fold0.csv \
  --artifact artifacts_rr_ft_k50_ep3 \
  --fold 0 --fixed-candidates \
  --output experiments/generated/variant_scores_fold0.parquet
```

Runner сейчас намеренно не создан: для него нужно импортировать внутренние функции подготовки passages и cross-encoder, а это уже отдельный исполняемый файл. Все остальные изменения в этом задании ограничены директорией `experiments/`.

In [12]:
# Эта ячейка только читает готовый результат внешнего GPU-runner, если он появился.
variant_score_path = EXPERIMENTS / 'generated' / f'variant_scores_fold{PROXY_FOLD}.parquet'
if not variant_score_path.exists():
    print('GPU-результата пока нет:', variant_score_path)
else:
    variant_scores = pd.read_parquet(variant_score_path)
    required = {'query_id', 'variant', 'article_id', 'score'}
    assert required <= set(variant_scores.columns), required - set(variant_scores.columns)
    metric_rows = []
    for (qid, variant), group in variant_scores.groupby(['query_id', 'variant']):
        predicted = group.sort_values(['score', 'article_id'], ascending=[False, True])['article_id'].astype(int).tolist()
        metric_rows.append({
            'query_id': int(qid), 'variant': variant,
            'ap@10': ap_at_10(predicted, ground_truth[int(qid)]),
            'recall@10': recall_at_k(predicted, ground_truth[int(qid)], 10),
            'first_rank': first_relevant_rank(predicted, ground_truth[int(qid)]),
        })
    variant_metrics = pd.DataFrame(metric_rows)
    original = variant_metrics[variant_metrics['variant'] == 'original'][['query_id', 'ap@10']].rename(columns={'ap@10': 'original_ap'})
    variant_metrics = variant_metrics.merge(original, on='query_id')
    variant_metrics['delta_ap'] = variant_metrics['ap@10'] - variant_metrics['original_ap']
    variant_summary = variant_metrics.groupby('variant').agg(
        n=('query_id', 'size'), map10=('ap@10', 'mean'), delta=('delta_ap', 'mean'),
        top10_miss=('first_rank', lambda x: (x > 10).mean()),
    ).reset_index().sort_values('delta', ascending=False)
    display(variant_summary.style.format({'map10': '{:.4f}', 'delta': '{:+.4f}', 'top10_miss': '{:.1%}'}))

,variant,n,map10,delta,top10_miss
2,mask_entities,82,0.6897,+0.0042,2.4%
5,topic_distractor,82,0.6860,+0.0005,4.9%
3,original,82,0.6855,+0.0000,2.4%
7,typo_keyboard,82,0.6851,-0.0004,4.9%
8,typo_swap,82,0.6739,-0.0116,1.2%
6,typo_delete,82,0.6704,-0.0151,2.4%
4,polite_noise,82,0.6627,-0.0228,1.2%
0,case_punct,82,0.6476,-0.0379,3.7%
1,drop_content,82,0.6476,-0.0379,2.4%
9,typo_two,82,0.6449,-0.0406,1.2%


## 8. Правила принятия решений

1. `case_punct` заметно ухудшает результат → сначала искать ошибку нормализации/контрактов, а не обучать новую модель.
2. Одна опечатка почти безвредна, а `typo_two` деградирует → система достаточно робастна для реальных запросов; дополнительный spell correction имеет низкий приоритет.
3. Исправленные естественные запросы стабильно лучше оригиналов → тестировать multiview `original + corrected`, не заменять original.
4. `polite_noise` или `topic_distractor` сильно вредят → перспективнее canonical intent / query rewrite, чем орфографический корректор.
5. `drop_content` выявляет несколько критичных слов → строить отчёт по удалённым токенам и проверять, соответствуют ли они реальному intent или случайным коррелятам.
6. Положительный общий эффект knownness при отрицательном cold-slice не принимать без оценки ожидаемого test distribution.
7. Однофолдовый результат — фильтр. Любой вариант, который планируется использовать, подтверждается на всех пяти OOF-фолдах paired-тестом.

Практический порядок: **error analysis → single-fold fixed-candidate robustness → full-path только для победителей → 5-fold confirmation одного варианта**.

---

## 9. Сводка результатов: что сделано и что дало (обновлено 2026-07-20)

Все OOF — честные dev-400, blend-голова; «паблик» — тестовая система (засчитывается лучшая попытка).

| Конфиг | dev-400 OOF MAP@10 | Паблик |
|---|---|---|
| FT labeled + LR-blend (до граф-фич) | 0.7333 | 0.690 |
| + лемматизация лексики (reuse ночных весов) | 0.7378 | — |
| + граф-фичи головы (`co_top_p` cond / link / sim / `gt_freq` / `co_x_margin`) | 0.7484 | — |
| свежий FT: top-30, 2 эпохи | 0.7422 | — |
| свежий FT: top-50, эпохи 1 / 2 / **3** | 0.7361 / 0.7514 / **0.7608** | **0.7094** |
| рефит-500 того же конфига (k50, ep3, resume) | OOF-500 → `artifacts_rr_ft_final` | следующая попытка |

Разрыв OOF→паблик похож между поколениями (0.7333→0.690 и 0.7608→0.7094, оба −0.04…−0.05).
Это совместимо с selection bias после многих dev-экспериментов и сдвигом test-distribution, но само по себе не доказывает конкретную причину и не используется как аргумент об отсутствии утечки.

### Проверено и отклонено — knownness-фичи из §4 (по правилу №6 из §8)

`log_freq` уже в голове (это `gt_freq`). Остальное — OOF-тест на `artifacts_rr_ft_k50_ep3`,
cold-slice = 25 запросов, у которых все GT unseen в train их фолда:

| Вариант | MAP OOF | cold-slice | p vs база |
|---|---|---|---|
| база lr_graph | 0.7608 | 0.484 | — |
| + все 4 фичи | 0.7603 | 0.465 | 0.92 |
| + только interactions | 0.7549 | 0.445 | 0.18 |
| + is_rare, is_unseen | 0.7622 | 0.453 | 0.77 |
| + unseen×dense | 0.7602 | 0.431 | 0.90 |
| + logfreq×margin | 0.7597 | 0.473 | 0.78 |

Ни один вариант не даёт значимого общего прироста, и все ухудшают cold-slice —
популяр-прайор уже выжат связкой `gt_freq` + co-фичи, пороговые копии лишь режут хвост.
(MLP-голова аналогично отклонена раньше — см. §10 graph-ноутбука: прирост = шум сида.)

### Скрининг вариантов запросов (§6–7): почва готова

- `experiments/generated/query_variants_fold0.csv` — 82 запроса fold-0 × 10 вариантов,
  сгенерирован детерминированно (sha256-сиды ⇒ бит-в-бит то, что дал бы `EXPORT_VARIANTS=True`);
- `experiments/score_query_variants.py` — GPU-runner **режима A** по контракту §7:
  кандидаты из `candidates/dev_oof.parquet`, пассажи зафиксированы по кэшу q_emb
  оригинального запроса (изоляция cross-encoder, dense-энкодер не загружается),
  модель `fold_0` из `artifacts_rr_ft_k50_ep3`. Режим B — осознанно только после A.
- Запуск после освобождения GPU (~30 мин на все 10 вариантов; smoke: `--limit 3`):

```bash
cd solution && CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src HF_HUB_OFFLINE=1 \
python3 experiments/score_query_variants.py \
  --variants experiments/generated/query_variants_fold0.csv \
  --artifact artifacts_rr_ft_k50_ep3 --fold 0 --fixed-candidates \
  --output experiments/generated/variant_scores_fold0.parquet
```

Результат подхватит ячейка §7 выше — просто перезапустить её.

### Бюджеты и протокол на финише

- **Попытки теста:** потрачено 2 (0.690, 0.7094), осталось 5; засчитывается лучшая →
  шлём всё, что локально не хуже; не тратим попытки на варианты, неотличимые даже по OOF.
- **Holdout:** 1/3 использовано; после сабмита 0.7094 паблик сам служит независимым
  подтверждением — оставшийся бюджет бережём.
- **Новые ветки учатся сразу на всех 500**: база — `configs/experiments/rerank_ft_full500.yaml`
  (скопировать, сменить `artifacts_dir`, убрать `resume_folds`). Отбор при этом идёт по OOF-500,
  куда входит бывший holdout — слегка оптимистичнее dev-400-протокола; осознанный размен ради
  скорости на финише, оговорка — в шапке конфига и в README. Арбитр — тестовая система.